<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook D05: Specialised Deep Learning Architectures</h2>
</div>

Worked solutions to the 2 exercises in
[Notebook D05: Specialised Deep Learning Architectures](../notebooks/D05_Specialised_architectures.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

The data, the rolling evaluation and `fit_and_score` from the notebook, with `horizon`, `max_steps` and
`seed` exposed as arguments so the exercises can vary them.

**This notebook fits ten models and takes around forty minutes on CPU.** Most of that is Exercise 1,
which deliberately repeats the same configurations across seeds.

In [ ]:
import sys
import importlib.util
import logging
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

NEURALFORECAST_AVAILABLE = importlib.util.find_spec("neuralforecast") is not None

if NEURALFORECAST_AVAILABLE:
    import torch
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NBEATS, NHITS

    torch.set_num_threads(1)
    logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
    logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
    warnings.filterwarnings("ignore")
    print("NeuralForecast is available.")
else:
    print("NeuralForecast is not installed. Run 'uv sync --group dl' to follow this notebook.")

ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

LOOKBACK = 168
MAX_STEPS = 300

n_observations = len(load)
TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

frame = pd.DataFrame({
    "unique_id": "AT",
    "ds": load.index,
    "y": load.values.astype(np.float32),
})
fitting_frame = frame.iloc[:train_end + VALIDATION_HOURS]


def rolling_origin(nf, model_name, horizon):
    """Forecast `horizon` steps at a time across the test period, without refitting."""
    actuals, predictions = [], []

    for origin in range(train_end + VALIDATION_HOURS, n_observations - horizon, horizon):
        forecast = nf.predict(df=frame.iloc[:origin])
        predictions.append(forecast[model_name].values[:horizon])
        actuals.append(frame["y"].values[origin:origin + horizon])

    return np.concatenate(actuals), np.concatenate(predictions)


def fit_and_score(model_class, model_name, horizon=24, max_steps=MAX_STEPS, seed=0):
    """Fit one model and score it over the test period."""
    started = time.time()

    nf = NeuralForecast(
        models=[model_class(h=horizon, input_size=LOOKBACK, max_steps=max_steps,
                            enable_progress_bar=False, random_seed=seed)],
        freq="h",
    )
    nf.fit(fitting_frame, val_size=VALIDATION_HOURS)

    actuals, predictions = rolling_origin(nf, model_name, horizon)

    return {
        "mae": mean_absolute_error(actuals, predictions),
        "seconds": time.time() - started,
        "parameters": sum(p.numel() for p in nf.models[0].parameters()),
    }


def naive_mae(horizon):
    """Repeat the most recent comparable block: yesterday for a day, last week for a week."""
    offset = 24 if horizon <= 24 else 168
    actuals, predictions = [], []

    for origin in range(train_end + VALIDATION_HOURS, n_observations - horizon, horizon):
        actuals.append(frame["y"].values[origin:origin + horizon])
        predictions.append(frame["y"].values[origin - offset:origin - offset + horizon])

    return mean_absolute_error(np.concatenate(actuals), np.concatenate(predictions))


if NEURALFORECAST_AVAILABLE:
    print(f"{len(frame):,} hourly rows, training on the first {train_end + VALIDATION_HOURS:,}")
    print(f"Naive baseline at h=24: {naive_mae(24):.1f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Both models here were trained at the workshop budget of 300 steps. Refit N-BEATS at 1000 steps and compare. Then do it again for three different `random_seed` values at each budget, and report the mean and the spread rather than single runs. Does more training help, and is one run enough to tell?

In [ ]:
if NEURALFORECAST_AVAILABLE:
    single = pd.Series({
        steps: fit_and_score(NBEATS, "NBEATS", max_steps=steps, seed=0)["mae"]
        for steps in (300, 1000)
    })

    print("One run each, seed 0:")
    print(single.round(1).to_string())
    print(f"\n1000 steps is {'better' if single[1000] < single[300] else 'WORSE'} "
          f"by {abs(single[1000] - single[300]):.1f} MW")

On one seed, **1000 steps scores 231.4 against 300 steps' 225.6 — the longer run is worse.**

That is a tidy, quotable result, and it is wrong. It is the answer a single run gives, and a single run
is not enough to tell. Notebook [D01](../notebooks/D01_Neural_networks_intro.ipynb) made this point about
a small MLP; it applies with more force here, and the exercise asks for the seeds precisely so that this
first table can be disbelieved.

In [ ]:
SEEDS = (0, 1, 2)

if NEURALFORECAST_AVAILABLE:
    records = [
        {"steps": steps, "seed": seed,
         "MAE": fit_and_score(NBEATS, "NBEATS", max_steps=steps, seed=seed)["mae"]}
        for steps in (300, 1000)
        for seed in SEEDS
    ]

    runs = pd.DataFrame(records).pivot(index="steps", columns="seed", values="MAE")
    runs["mean"] = runs.mean(axis=1)
    runs["sd"] = runs[list(SEEDS)].std(axis=1)

runs.round(1)

**Across three seeds the ordering reverses: 1000 steps averages 211.3 MW against 300 steps' 225.9. More
training does help, by about 6.5%.**

And the reason one run could not tell you is in the last column:

| budget | runs | mean | sd |
|---|---|---|---|
| **300 steps** | 225.6, 227.3, 224.8 | 225.9 | **1.2** |
| **1000 steps** | 231.4, 200.8, 201.7 | 211.3 | **17.4** |

**The long budget is fourteen times more variable than the short one.** Two of its three runs land near
201, comfortably beating anything the short budget produces; the third lands at 231.4, comfortably worse
than all of them. Seed 0 happened to be that third one.

This is worth more than the accuracy finding, because it is a fact about how to run the experiment rather
than about this model:

**Training longer increases spread as well as lowering the mean.** A short budget stops every run at
roughly the same under-trained place, so the runs agree with each other — they are reliably mediocre. A
long budget lets each run follow its own trajectory into a different part of the loss surface, and those
places differ in quality. The mean improves; the guarantee does not.

**So a single long run is a coin flip and a single short run is not.** The practical consequence is
awkward: the configuration you most want to evaluate with one run is the one where one run tells you
least. If you are comparing budgets, or anything else that changes how long a network trains, you have to
average over seeds or you are measuring the seed.

Two footnotes:

- **This is why the notebook's own numbers are quoted from a fixed configuration.** The 225.6 in section 5
  is seed 0 at 300 steps, and it reproduces exactly because both are pinned. Reproducibility is not the
  same as reliability: a number can be perfectly reproducible and still be a poor estimate of what the
  method does.
- **The comparison in section 5 is conservative, as the notebook claims, but for a subtler reason than it
  gives.** N-BEATS at 300 steps is not simply "on a smaller budget" — it is on the budget where its score
  is most trustworthy. Its 225.6 against the TCN's 260.6 would only widen with more training, on average,
  while becoming harder to quote.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> The table in section 7 says to reach for N-HiTS when the horizon runs to hundreds of steps. Test it: refit both models with `h=168`, a week ahead instead of a day, and score them against a baseline that repeats the previous week. Does the ordering change, and is either model worth having at that horizon?

In [ ]:
if NEURALFORECAST_AVAILABLE:
    rows = []
    for horizon in (24, 168):
        for model_class, name in [(NBEATS, "NBEATS"), (NHITS, "NHITS")]:
            outcome = fit_and_score(model_class, name, horizon=horizon)
            rows.append({"horizon": horizon, "model": name, "MAE": outcome["mae"],
                         "parameters": outcome["parameters"], "seconds": outcome["seconds"]})
        rows.append({"horizon": horizon, "model": "naive", "MAE": naive_mae(horizon),
                     "parameters": 0, "seconds": 0.0})

    horizons = pd.DataFrame(rows).pivot(index="horizon", columns="model", values="MAE")
    horizons["N-HiTS advantage"] = horizons["NBEATS"] - horizons["NHITS"]
    horizons["best model vs naive"] = (
        1 - horizons[["NBEATS", "NHITS"]].min(axis=1) / horizons["naive"]
    )

horizons.round(3)

**The ordering does change, and the improvement over the baseline collapses.**

At a day ahead N-BEATS leads by 10.5 MW, 225.6 against 236.1. At a week ahead N-HiTS leads by 3.7,
404.0 against 407.7. So the claim in section 7 is directionally confirmed: N-HiTS's relative position
improves as the horizon lengthens.

But calling that "N-HiTS wins" would be overstating a 0.9% difference, and Exercise 1 supplies exactly the
yardstick needed to say so. **Run-to-run spread at 300 steps was 1.2 MW at the short horizon**, and a
3.7 MW gap is only about three times that — on one seed each. The 10.5 MW gap at h=24 is eight times the
noise and is safe to believe; the 3.7 MW gap at h=168 is not. The honest statement is that **N-BEATS's
advantage disappears at the longer horizon**, not that it reverses.

The second question has a blunter answer.

In [ ]:
if NEURALFORECAST_AVAILABLE:
    fig, ax = plt.subplots(figsize=(10, 5))

    width = 0.26
    positions = np.arange(2)
    for offset, (column, colour) in enumerate(
        [("NBEATS", "seagreen"), ("NHITS", "mediumseagreen"), ("naive", "crimson")]
    ):
        ax.bar(positions + (offset - 1) * width, horizons[column], width,
               label=column, color=colour)
        for x, value in zip(positions + (offset - 1) * width, horizons[column]):
            ax.text(x, value + 6, f"{value:.0f}", ha="center", fontsize=9)

    ax.set_xticks(positions)
    ax.set_xticklabels(["24 hours ahead", "168 hours ahead"])
    ax.set_title("What a week-ahead horizon costs", fontsize=13, fontweight="bold")
    ax.set_ylabel("MAE (MW)")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**At a week ahead, neither model is worth much.** The best of them scores 404.0 against a baseline of
424.2 — an improvement of **4.8%**, where at a day ahead the same models beat their baseline by **60%**.

That is the finding that matters, and it dwarfs the question the exercise asked. Choosing between N-BEATS
and N-HiTS at this horizon is arguing about 0.9% inside a result that is barely distinguishable from
repeating last week.

The reason is not architectural. Notebook [D02](../notebooks/D02_Recurrent_networks.ipynb) measured the
same thing directly: error grows with horizon because the recent trajectory stops being informative, while
the naive forecast's error is flat because "the same hour last week" is equally good, or equally bad, at
any distance. Extend the horizon far enough and every model converges on the baseline from above. A week
ahead is already most of the way there for Austrian load.

So the practical reading of section 7's table is narrower than it looks. **"Reach for N-HiTS when the
horizon runs to hundreds of steps" is advice about which model to pick, and the prior question is whether
to pick one at all.** If a week-ahead forecast is what you need, the answer on this series is that 5% over
a one-line baseline does not justify a 3-million-parameter model, and the effort belongs in finding
information the model does not have — weather forecasts above all — rather than in choosing between two
architectures that are both out of road.

One honest caveat on the comparison itself: 168 steps is not "hundreds", and N-HiTS is designed for
horizons where its multi-rate sampling has many resolutions to work across. This test sits at the bottom
of the range the advice is about, which is consistent with finding the effect present but small. It also
did not come out cheaper here — N-HiTS took longer to fit than N-BEATS at both horizons — but on a single
CPU thread that says more about the machine than the architecture, as the notebook cautions about every
timing in Part D.

---

Back to [Notebook D05](../notebooks/D05_Specialised_architectures.ipynb), or on to
[Notebook E01](../notebooks/E01_End_to_end_pipeline.ipynb).